# 01 · Convert a detector: Super-K from its GDML

lumen does not read GDML (the geometry file Geant4 reads) while it simulates. It reads a **detector file**: the
geometry, materials, optical surfaces and light sensors, converted once and checked. This notebook makes Super-K's.

A GDML file says what Geant4 builds, but not everything lumen needs: which volume is the detector, which volumes
are sensors and how they read light. That is the **recipe**, which lumen-wc keeps for each WCSim detector
(`lumen_wc.wcsim`).

In [ ]:
import os
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

import lumen
import lumen_wc
from lumen.compare import references

WORK = Path(os.environ.get("LUMEN_NOTEBOOKS", "lumen-notebooks"))
WORK.mkdir(parents=True, exist_ok=True)

entry = lumen_wc.DETECTORS["sk"]  # a lumen.DetectorEntry
recipe, gdml, inputs = entry.recipe, entry.gdml, entry.inputs
print("GDML:", gdml)
print("detector volume:", recipe.detector_volume)
for g in recipe.recipe.groups:  # the readout gate is set at conversion, from the GDML (below)
    print(f"sensor group {g.name!r}: volumes matching {g.lv_pattern!r}, counted by rule {g.rule!r}, "
          f"read with value set {g.values!r}")

## Convert

`lumen.convert("lumen_wc:sk", ...)` reads that entry of `lumen_wc.DETECTORS` (a `lumen.DetectorEntry`: the recipe, the GDML and the
inputs, under the reference data directory `references.store()`: `LUMEN_DATA`, else lumen's cache; a GDML it lacks
is fetched and checked first) and converts it. On the way lumen applies its
general import rules, checks the structure, and runs the **placement check**: Geant4 assumes that volumes placed
side by side never overlap and that a volume lies inside its mother; lumen tests both on points just inside every
surface (what Geant4's `CheckOverlaps` does) and records what it finds. It never changes the geometry.

In [ ]:
path = WORK / "sk.lumen"
t0 = time.perf_counter()
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    lumen.convert("lumen_wc:sk", out=path, data=references.store(), overwrite=True)
print(f"converted in {time.perf_counter() - t0:.0f} s to {path}")
print("readout gate:", lumen_wc.wcsim.readout_gate(references.store() / gdml), "ns after emission")
for w in caught:
    if "placement" in str(w.category.__name__).lower():
        print("placement warning:", w.message)

## The geometry and overlap checks

The conversion's report lists every decision and the placement check's verdict. Super-K has no overlapping volumes.

In [ ]:
det = lumen.load(path)
for line in det.file.report:
    print("-", line)

The **geometry check** goes further: it asks Geant4 itself where random points are (inside which volume, which
material) and compares with lumen's answer, point by point. It needs a Geant4 build with the `g4ref` probe, which
you point to with the environment variable `G4REF`; without it this cell only says so. Every point outside the
opaque meshes must agree.

In [ ]:
g4ref = os.environ.get("G4REF")
if g4ref:
    from lumen.compare import geometry

    rows = geometry.validate("sk", detector=path, g4ref=g4ref, extra=[lumen_wc.REFERENCES])
    for r in rows:
        print(r["quantity"], r["value"], "pass" if r["pass"] else "FAIL", r["extra"])
else:
    print("G4REF is not set: run  python -m lumen.compare geometry sk --package lumen_wc --detector-file", path,
          "--g4ref BINARY")

## What the file holds

A detector file is a directory of plain JSON and arrays. The loaded `Detector` summarises it:

In [ ]:
print(det)
print("files:", sorted(p.name for p in path.iterdir()))
print("materials:", list(det.file.materials))
print("optical surfaces:", list(det.file.surfaces))
print("converted from:", det.file.provenance["source"])

Every number lumen uses, except the geometry, is a **parameter** with a name, a unit, an allowed range and a
*gradient method* (how lumen differentiates it, notebook 03). `explain` says where the value came from.

In [ ]:
p0 = det.params()
print(len(p0), "parameters, for example:")
for key in ("materials/Water/ABSLENGTH", "materials/Water/RAYLEIGH", "surfaces/BSSkinSurface/REFLECTIVITY",
            "sensors/pmt/efficiency"):
    info = p0.info(key)
    print(f"  {key:38s} unit {info.unit or '-':3s} range {info.range}  method {info.method}  from {p0.explain(key)}")

Optical properties are tables against photon energy. Here are the water's absorption and scattering lengths
against wavelength (wavelength in nm = 1239.84 / energy in eV).

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
for key, label in (("materials/Water/ABSLENGTH", "absorption length"), ("materials/Water/RAYLEIGH",
                                                                         "Rayleigh scattering length")):
    t = p0[key]
    ax.plot(1239.84 / np.asarray(t.energy), np.asarray(t.value), label=label)
ax.set(xlabel="wavelength (nm)", ylabel="length (m)", yscale="log", title="Super-K water (WCSim's tables)")
ax.legend()
plt.show()

And the sensors, one per PMT, in the order every result uses (`light.arrivals[i]` is sensor *i*):

In [ ]:
table = det.sensor_table()
print(len(table.name), "sensors; the first:", table.name[0])
print("position (m):", table.position[0], " facing:", table.axis[0])

**Next:** 02 compares lumen with Geant4 and WCSim on this detector.